# **2A_veri_hazirlama**

# **2.1 Gerekli Kutuphanelerin Yuklenmesi**

### Neden Onemli?
Veri on isleme asamasinda kullanilacak tum fonksiyonlari, grafik araclarini ve veri donusum araclarini bu kutuphaneler saglar.

### Bu bolumde ne yapacagiz?
- **`NumPy, pandas, seaborn, sklearn`** gibi paketleri import edecegiz.
- **Veri on isleme** sirasinda kullanacagimiz fonksiyonlari tanimlayacagiz.

In [ ]:
import numpy as np                                                  # NumPy: Matematiksel islemler icin kullanilir (ornegin: NaN degerleri, diziler)
import pandas as pd                                                 # pandas: Tablo seklindeki verileri okumak, islemek ve analiz etmek icin kullanilir
import seaborn as sns                                               # seaborn: Guzel ve anlasilir grafikler cizmek icin kullanilir
import matplotlib.pyplot as plt                                     # matplotlib: Grafik cizim kutuphanesi (seaborn'un altyapisini olusturur)
from matplotlib.cbook import boxplot_stats                          # boxplot_stats: Kutu grafigi icin istatistik degerleri hesaplar
import random                                                       # random: Rastgele sayi uretmek icin Python'un yerlesik modulu
from random import sample                                           # sample: Bir listeden tekrarsiz rastgele eleman secmek icin kullanilir
from sklearn.model_selection import train_test_split                # train_test_split: Veriyi egitim ve test setine otomatik bolmek icin kullanilir
from sklearn.preprocessing import MinMaxScaler, StandardScaler     # MinMaxScaler: Verileri 0-1 arasina sikistirmak icin, StandardScaler: Ortalama=0, std=1 yapmak icin


print("Gerekli kutuphaneler basariyla yuklendi.")

---

---

---

# **2.2 Veri Okuma**

### Neden Onemli?
Veriyi dogru bicimde okumak ve bir **`DataFrame`** haline getirmek, sonraki tum veri on isleme ve analiz adimlari icin kritik bir adimdir. Eger veri yanlis bir konumdan okunursa veya sutun basliklari hatali tanimlanirsa, ileriki asamalarda eksik ya da hatali sonuclar alabiliriz.

### Bu bolumde ne yapacagiz?
- Google Drive'i baglayarak **(`mount`)** veri setinin bulundugu klasore erisecegiz.

- **`automobile.csv`** dosyasini, pandas'in **`read_csv`** fonksiyonunu kullanarak okuyacagiz.
- Veri setinin sutun veri tiplerini **(`dtypes`)** inceleyecegiz.
- Ayrica **`head()`** fonksiyonuyla veri icerigine hizlica goz atacagiz.

In [ ]:
# 1) Google Drive'i mount ediyoruz (Colab'da dosyalarimiza erismek icin Drive'i baglamamiz gerekir)
from google.colab import drive                                      # google.colab modulu sadece Google Colab ortaminda calisir
drive.mount('/content/drive', force_remount=True)                   # Drive'i '/content/drive' yoluna baglar, force_remount=True her seferinde yeniden baglanir

# 2) 'automobile.csv' dosyasini, Colab Notebooks klasorunden tam yol ile okuyoruz
veriSeti = pd.read_csv(                                             # pd.read_csv(): CSV dosyasini okuyup bir DataFrame (tablo) olusturur
    "/content/drive/MyDrive/Colab Notebooks/automobile.csv"         # Dosyanin Google Drive'daki tam yolu
)

# 3) Veri setinin boyutu ve sutun veri tipleri
print("Veri Seti Boyutu:", veriSeti.shape)                          # shape: (satir sayisi, sutun sayisi) seklinde boyutu gosterir
print("\nSutun Veri Tipleri:\n", veriSeti.dtypes)                   # dtypes: Her sutunun veri tipini gosterir (int64, float64, object vb.)
display(veriSeti.head(10))                                          # head(10): Tablonun ilk 10 satirini ekrana gosterir

---

---

---

# **2.3 Eksik Veri Tamamlama (Missing Data Imputation)**

### Neden Onemli?
Eksik veriler, modelin dogrulugunu olumsuz etkileyebilir veya hata olusturabilir. Bu nedenle eksik verileri uygun yontemlerle doldurmak **(`imputation`)**, veri kalitesini artirir ve analiz veya makine ogrenmesi modellerinden daha guvenilir sonuclar elde etmemize yardimci olur.

### Bu bolumde ne yapacagiz?
- **`normalized_losses`** sutunundaki **'?'** karakterlerini **`NaN`'e** donusturecegiz.
- **`horsepower`** ve **`peak_rpm`** sutunlarindaki **'?'** karakterlerini de **`NaN`'e** donusturecegiz.
- Bu sutunlarin veri tipini sayisal **`(float64)`** haline getirecegiz.
- Eksik veri sayisini inceleyip, ortalama degerle dolduracagiz.
- **`fillna()`** metodu kullanimini ornek olarak gorecegiz.

In [ ]:
# On kontrol: '?' karakteri iceren sutunlari tespit edelim
for col in veriSeti.columns:                                        # veriSeti.columns: Tum sutun isimlerini listeler
    soru_isareti = (veriSeti[col] == "?").sum()                     # Her sutunda '?' olan hucrelerin sayisini hesaplar
    if soru_isareti > 0:                                            # Eger '?' sayisi 0'dan buyukse ekrana yazdir
        print(f"{col} sutununda '?' sayisi: {soru_isareti}")

# 1) '?' karakterini NaN'e donusturme (tum ilgili sutunlar icin)
eksik_sutunlar = ["normalized_losses", "horsepower", "peak_rpm"]    # Icinde '?' bulunan sutunlari bir listeye topluyoruz
for col in eksik_sutunlar:                                          # Her bir sutun icin dongu baslatiyoruz
    veriSeti.loc[veriSeti[col] == "?", col] = np.nan                # loc[kosul, sutun]: '?' olan hucreleri bulup NaN (bos deger) ile degistir

# 2) Ilgili sutunlarin tipini float64'e donusturme
for col in eksik_sutunlar:                                          # Ayni sutunlar icin tekrar dongu
    veriSeti[col] = veriSeti[col].astype("float64")                 # astype("float64"): Sutunu metin tipinden ondalikli sayi tipine cevirir

# 3) Eksik veri sayisini (ilk kontrol) ekrana bastirma
print("\nEksik Veri Sayisi (Once):\n", veriSeti.isnull().sum())     # isnull().sum(): Her sutundaki NaN (bos) degerlerin sayisini verir

# 4) fillna() metodu ile ortalama ile doldurma
for col in eksik_sutunlar:                                          # Eksik deger iceren her sutun icin
    veriSeti[col] = veriSeti[col].fillna(veriSeti[col].mean().round(0))  # fillna(): NaN degerleri, sutunun ortalamasiyla doldurur. round(0): tam sayiya yuvarlar

print("\nEksik Veri Sayisi (Sonra):\n", veriSeti.isnull().sum())    # Doldurma isleminden sonra tekrar kontrol: Artik 0 olmali

---

---

---

# **2.4 Veri Ayiriklastirma (Binning / Discretization)**

### Neden Onemli?
- Surekli (sayisal) veriler, kategorik (gruplandirilmis) degerlere donusturulur.
- Bu donusumle, veriye yeni bir sutun eklenir veya mevcut sutun siniflara ayrilir.
- Amac: Sayisal degerleri daha anlamli ve genellenmis kategorilere indirgemektir.

Ornegin: `city_mpg` degiskeni;
- **21'in altinda** ise -> **Dusuk**
- **21 - 30 arasi** ise -> **Orta**
- **30 ve ustu** ise -> **Yuksek** olarak siniflandirilabilir.

Bu islem sonucunda:
- Veri setine **yeni bir kategorik sutun (ornegin: `durum`)** eklenir.
- Satir sayisi degismez; sadece her satir bir kategoriye atanir.

---

### Bu bolumde ne yapacagiz?
- `city_mpg` sutununu **Dusuk**, **Orta** ve **Yuksek** kategorilerine ayiracagiz.
- Uc farkli yontemle ayiriklastirma islemi gerceklestirecegiz:

**1.** **`pd.cut()`** fonksiyonu ile **sabit aralik (fixed interval)** yontemi uygulayacagiz.
**2.** **`pd.qcut()`** fonksiyonu ile **esit frekans (quantile)** yontemi kullanacagiz.
**3.** **`pd.cut()`** fonksiyonu ile **esit genislik (equal interval)** yontemini uygulayacagiz.


Her yontemde amac, `city_mpg` degerlerini farkli siniflandirma stratejileriyle gruplandiirmak ve analizlerde kullanilabilecek kategorik yapilar olusturmaktir.

In [ ]:

# 1.Yontem: Fixed (sabit) araliklarla bolme
bolmeKategorileri = ["Dusuk", "Orta", "Yuksek"]                    # Olusturulacak kategori isimleri
bolmeler = [12, 20.9, 29.9, 50]                                    # Sinir degerleri: 12-20.9=Dusuk, 20.9-29.9=Orta, 29.9-50=Yuksek
veriSeti["durum"] = pd.cut(                                         # pd.cut(): Sayisal degerleri belirlenen araliklara gore keser ve etiketler
    veriSeti["city_mpg"],                                           # Hangi sutunu bolecegimizi belirtiyoruz
    bins=bolmeler,                                                  # bins: Bolme sinirlarini tanimlar
    labels=bolmeKategorileri                                        # labels: Her araliga verilecek isimler
)
print("Durum (fixed cut) dagilim:\n", veriSeti.durum.value_counts())
print(f"\n{'-'*82}\n")

# 2.Yontem: Equal Frequency (qcut)
durum_ef = pd.qcut(veriSeti["city_mpg"], q=3)                      # pd.qcut(): Verileri esit sayida gozlem icerecek sekilde 3 gruba boler

print("Durum (Esit frekans) dagilim:\n", durum_ef.value_counts())
print(f"\n{'-'*82}\n")

# 3.Yontem: Equal Interval (esit genislik)
durum_ea = pd.cut(veriSeti["city_mpg"], bins=3)                     # pd.cut(bins=3): Deger araligini esit genislikte 3 parcaya boler
print("Durum (Esit genislik) dagilim:\n", durum_ea.value_counts())
print(f"\n{'-'*82}\n")

---

---

---

# **2.5 Veri Seti Ozeti (describe)**

### Neden Onemli?
Veri setinin genel istatistiksel ozelliklerini (ortalama, std, min, max vb.) gormek, veri hakkinda hizlica fikir sahibi olmamizi saglar. Hem sayisal hem de kategorik sutunlarin ozetini almak, veri analizinin sonraki adimlarini planlamada bize rehberlik eder.

### Bu bolumde ne yapacagiz?
- pd.set_option("display.max_columns", 20) ayariyla, DataFrame ekrana basilirken en fazla kac sutun gosterilecegini belirleyecegiz.

- veriSeti.describe(include="all") fonksiyonunu kullanarak, tum sutunlarin (sayisal ve kategorik) temel istatistiksel ozetini goruntuleyecegiz.

In [ ]:
# 1) pd.set_option ile ekrana basilacak sutun sayisini 20 ile sinirlandiriyoruz
pd.set_option("display.max_columns", 20)                            # Cok sutunlu tablolarda hepsinin gorunmesini saglar (varsayilan sinir dusuktur)

# 2) describe(include="all") ile sayisal ve kategorik tum sutunlarin ozet istatistiklerini aliyoruz
print("Veri Seti Ozeti (describe, include='all'):\n", veriSeti.describe(include="all"))  # describe(): Ortalama, std, min, max gibi temel istatistikleri gosterir

# **2.6 Veri Gruplama (Aggregate)**

### Neden Onemli?
Makine ogrenmesi ve veri bilimi sureclerinde, bazen tum veri setini tek bir butun olarak degerlendirmek yerine, verileri belirli ozelliklere gore alt gruplara ayirip ayri ayri incelemek faydalidir. Bu islem sayesinde her bir alt grup hakkinda istatistiksel ozet bilgiler elde edebiliriz. Bu, veriyi anlamak, farkli alt gruplari karsilastirmak ve karar vermek acisindan onemlidir.

**Ornegin:** Bir otomobil verisinde araclarin sehir ici yakit tuketimini (city_mpg) "durum" kategorisine (Dusuk, Orta, Yuksek) gore ayri ayri inceleyebilirsiniz. Boylece her grubun yakit tuketim ozelliklerini ayri ayri degerlendirebilirsiniz.


### **Veri Ayiriklastirma ile Gruplama Arasindaki Fark**

**Veri Ayiriklastirma (Discretization) = sayisali kategorik hale getirme**
- **Amac:** Surekli degiskenleri kategorik hale getirerek siniflandirmak.
- **On kosul:** Kategorik bir sutun yoksa, analiz icin kendimiz uretmeliyiz.

**Veri Gruplama (Aggregation) = zaten var olan kategorik degiskene gore istatistik cikarmak**
- **Amac:** Var olan kategorik sutuna gore gruplar olusturup bu gruplar icin istatistikler hesaplamak.
- **On kosul:** Veri setinde zaten kategorik bir degisken (ornegin durum) varsa dogrudan kullanilabilir.



### Bu bolumde ne yapacagiz?
- **groupby("durum")** ifadesini kullanarak, durum sutunu bazinda veri setini gruplara ayiracagiz.

- Farkli toplulastirma fonksiyonlari **(mean, sum, count, min, max, std, describe)** ile her bir kategorinin temel istatistiklerini goruntuleyecegiz.

In [ ]:
# 1. Adim: "durum" kategorisine gore "city_mpg" degiskeninin temel istatistiklerini cikar
df_ozet = veriSeti.groupby("durum", observed=False)["city_mpg"].describe().round(2)  # groupby(): Veriyi "durum" sutununa gore gruplar, describe(): Her grubun istatistiklerini hesaplar

# 2. Adim: 'durum' degiskeni indeks olarak degil, sutun olarak gorunsun diye resetlenir
df_ozet.reset_index(inplace=True)                                   # reset_index(): Grup isimlerini normal bir sutuna cevirir, inplace=True: Degisikligi kalici yapar

# 3. Adim: Jupyter defteri kullaniliyorsa tablo seklinde gorsellestirir
display(df_ozet)                                                    # display(): Tabloyu guzel formatli sekilde ekranda gosterir (sadece Jupyter/Colab'da calisir)

## `durum` Degiskenine Gore `city_mpg` (Sehir Ici Yakit Verimliligi) Istatistikleri

Asagidaki tablo, `city_mpg` (sehir ici mil per galon) sayisal degiskeninin, `durum` adli kategorik degiskene gore gruplandirilarak (aggregation) temel istatistiklerle ozetlenmesini gostermektedir.

---

### Yorum:

1. **Yuksek** `durum` kategorisi:
   - En yuksek **ortalama** `city_mpg` degeriyle en verimli gruptur.
   - Bu grupta yuksek yakit verimliligi olan araclar bulunur.

2. **Orta** kategori:
   - Orta duzeyde yakit verimliligi gosteren araclari icerir.
   - Degerler belirli bir aralikta yogunlasmistir.

3. **Dusuk** kategori:
   - Ortalama deger en dusuk olan gruptur.
   - Gruptaki araclar arasinda onemli cesitlilik olabilir.

---
---

---

---

---

# **2.7 Tekrarlayan Satirlarin Bulunmasi (Duplicated Rows)**

### Neden Onemli?
Tekrarlayan **(duplicate)** satirlar, veri analizi sonuclarini yaniltabilir. Ayni gozlemin veri setinde birden fazla kez yer almasi, ortalama ve diger istatistiksel hesaplamalari etkileyerek hatali sonuclara yol acabilir. Bu nedenle tekrarlayan satirlari tespit edip, gereksiz olanlari temizlemek veri kalitesini artirir.

### Bu bolumde ne yapacagiz?
- `duplicated()` fonksiyonunu kullanarak tekrarlayan satirlari bulacagiz.
- `drop_duplicates()` ile tekrarlayanlari temizleyecegiz.
- Tekrarlayan satirlari bulmak icin farkli keep parametreleri **(ornegin "first", "last")** deneyecegiz.

In [ ]:
# 1) duplicated() ile tekrarlayan satirlari tespit ediyoruz
tekrarlar_f = veriSeti.duplicated(keep="first")                     # keep="first": Ilk tekrari tutar (False), sonrakileri True isaretler
tekrarlar_l = veriSeti.duplicated(keep="last")                      # keep="last": Son tekrari tutar (False), onceki tekrarlari True isaretler

# 2) Herhangi birinin True oldugu satirlar, tekrarlayan satirlardir
indislerim = tekrarlar_f | tekrarlar_l                              # | (veya): Iki kosuldan en az biri True ise o satir tekrar eden satirdir

# 3) Tekrarlayan gozlemleri ekrana basalim
print("Tekrarlayan Gozlemler:\n", veriSeti[indislerim])             # veriSeti[indislerim]: Sadece True olan (tekrarlayan) satirlari filtreler

# 4) drop_duplicates() ile veri setinden tekrar eden satirlari kaldiriyoruz
veriSeti2 = veriSeti.drop_duplicates()                              # drop_duplicates(): Tekrarlayan satirlari siler, her satirdan sadece birini birakir

# 5) Kaldirma isleminden sonra veri setinin boyutunu (shape) inceleyerek degisikligi gozlemleyebiliriz
print("\nTekrarlayanlardan arindirilmis veriSeti2 shape:", veriSeti2.shape)  # Satir sayisi azaldiysa tekrarlayan satirlar vardi demektir

# **2.8 Aykiri Deger Analizi (IQR ve Boxplot)**

### Aykiri Deger Nedir?

Aykiri deger, veri setindeki diger degerlerden **belirgin sekilde uzak** olan gozlemlerdir.
Bu degerler cogu zaman hatali girisler, olcum hatalari veya sira disi durumlari temsil edebilir.

> Ornek: Eger `horsepower` sutunundaki degerlerin cogu 50-150 arasinda ise, 200 uzerinde bir deger aykiri olabilir.

---

### Neden Onemlidir?

- Aykiri degerler analizlerin sonucunu **yaniltabilir**.
- Ortalama, standart sapma gibi istatistikleri bozar.
- Bazi algoritmalar (ornegin KNN, k-means) aykiri degerlerden cok etkilenir.

---

### Aykiri Deger Nasil Bulunur?

**1. Gorsel yontem:** Kutu grafigi (boxplot)
**2. Istatistiksel yontem:** IQR (Interquartile Range) yontemi
Bu derste, `horsepower` degiskenini ornek alarak hem istatistiksel hem gorsel yontemle aykiri degerleri belirleyecegiz.

In [ ]:
# 1. Kartil (Q1) ve 3. Kartil (Q3) degerlerini hesaplayalim
q1 = veriSeti["horsepower"].quantile(0.25)                          # quantile(0.25): Verinin alt %25'lik degerini bulur (Q1 = 1. ceyrek)
q3 = veriSeti["horsepower"].quantile(0.75)                          # quantile(0.75): Verinin ust %75'lik degerini bulur (Q3 = 3. ceyrek)

# IQR (ceyrekler arasi aciklik): Q3 - Q1
IQR = q3 - q1                                                       # IQR: Verinin orta %50'sinin yayildigi aralik, aykiri deger tespitinde kullanilir

# Alt ve ust sinirlari belirliyoruz
alt = q1 - 1.5 * IQR                                                # Alt sinir: Bu degerin altindaki her sey aykiri deger sayilir
ust = q3 + 1.5 * IQR                                                # Ust sinir: Bu degerin ustundeki her sey aykiri deger sayilir

print(f"Aykiri deger alt siniri: {alt}, ust siniri: {ust}")          # Hesaplanan alt ve ust sinirlari ekrana yazdirir

In [ ]:
# Aykiri olan satirlarin indekslerini buluyoruz
ust_aykiriDegerInd = np.where(veriSeti["horsepower"] >= ust)[0]     # np.where(): Kosulu saglayan satirlarin indeks numaralarini dondurur
alt_aykiriDegerInd = np.where(veriSeti["horsepower"] <= alt)[0]     # Alt sinirin altindaki degerlerin indekslerini bulur

# Ust sinirin uzerindeki aykiri degerleri indeks : deger formatinda yazdiriyoruz
print("Ust sinirin uzerindeki aykiri degerler:")
for i in ust_aykiriDegerInd:                                        # Her bir aykiri degerin indeksini tek tek gezeriz
    print(f"Indeks: {i}, Deger: {veriSeti['horsepower'][i]}")       # f-string: Degiskenleri metin icine yerlestirmek icin kullanilir

# Alt sinirin altindaki aykiri degerleri indeks : deger formatinda yazdiriyoruz
print("\nAlt sinirin altindaki aykiri degerler:")
for i in alt_aykiriDegerInd:                                        # Alt sinir icin de ayni islemi tekrarlariz
    print(f"Indeks: {i}, Deger: {veriSeti['horsepower'][i]}")

### Kutu Grafigi

- **Kutu grafigi (boxplot)**, verinin dagilimini ve olasi aykiri degerleri gorsel olarak anlamamizi saglar.
- Orta cizgi: **Ortanca (medyan)**
- Kutunun alt ve ust kenarlari: **1. ve 3. ceyrek (Q1, Q3)**
- Alt ve ust "biyik" cizgileri:
  - Alt sinir = Q1 - 1.5 * IQR
  - Ust sinir = Q3 + 1.5 * IQR
- Bu sinirlarin **disindaki daireler**, **aykiri degerleri** gosterir.

Bu grafik sayesinde:
- Hangi degerlerin aykiri oldugunu gorsellestirebiliriz.
- Alt sinirin altinda aykiri deger olup olmadigini,
- Ust sinirin uzerinde ne kadar aykiri deger oldugunu kolayca gozlemleyebiliriz.

In [ ]:
# Basit Kutu Grafigi

import seaborn as sns                                               # seaborn kutuphanesini tekrar import ediyoruz (guvenlik icin)
import matplotlib.pyplot as plt                                     # matplotlib grafik cizim kutuphanesi

plt.figure(figsize=(8, 5))                                          # figsize=(genislik, yukseklik): Grafik boyutunu inç cinsinden ayarlar
sns.boxplot(y="horsepower", data=veriSeti)                          # boxplot(): horsepower sutunu icin kutu grafigi cizer (aykiri degerler daire olarak gorunur)
plt.title("Horsepower Niteligne Ait Kutu Grafigi")                  # title(): Grafigin basligini belirler
plt.show()                                                          # show(): Grafigi ekranda gosterir

In [ ]:
# Aciklamali Kutu Grafigi

plt.figure(figsize=(10, 7))                                         # Daha buyuk bir grafik alani olustur

# Aykiri degerleri gizleyerek kutu grafigi ciz
sns.boxplot(y="horsepower", data=veriSeti, showfliers=False)        # showfliers=False: Aykiri degerleri kutu grafiginde gizler (biz kendimiz ekleyecegiz)
plt.title("Gercek Verilerle Horsepower Degiskenine Ait Aciklamali Kutu Grafigi")

# Istatistik degerleri
ortalama = veriSeti["horsepower"].mean()                            # mean(): Sutunun aritmetik ortalamasini hesaplar
medyan = veriSeti["horsepower"].median()                            # median(): Verileri siralayinca ortaya dusen degeri bulur
q1 = veriSeti["horsepower"].quantile(0.25)                          # 1. ceyrek (alt %25)
q3 = veriSeti["horsepower"].quantile(0.75)                          # 3. ceyrek (ust %25)
iqr = q3 - q1                                                       # Ceyrekler arasi aciklik
alt = q1 - 1.5 * iqr                                                # Aykiri deger alt siniri
ust = q3 + 1.5 * iqr                                                # Aykiri deger ust siniri

# Ortalama (turuncu)
plt.plot([0], [ortalama], "o-", color="orange")                     # plot(): Grafik uzerine bir nokta koyar ("o-" = daire isareti)
plt.text(0.1, ortalama, f"Ortalama:\n{ortalama:.2f}", color="orange", fontsize=10)  # text(): Noktanin yanina aciklama yazisi ekler

# Medyan (siyah nokta)
plt.plot(0, medyan, "ko")                                           # "ko" = siyah (k=black) daire (o=circle)
plt.text(-0.35, medyan, f"Medyan\n{medyan:.2f}", fontsize=10)

# Q1
plt.plot(0, q1, "ks")                                               # "ks" = siyah kare (s=square)
plt.text(-0.35, q1, f"Q1 (Alt Ceyrek)\n{q1:.2f}", fontsize=9)

# Q3
plt.plot(0, q3, "ks")
plt.text(-0.35, q3, f"Q3 (Ust Ceyrek)\n{q3:.2f}", fontsize=9)

# Alt sinir (mavi kare)
plt.plot(0, alt, "bs")                                               # "bs" = mavi (b=blue) kare (s=square)
plt.text(0.15, alt + 2, f"Alt Sinir\n{alt:.2f}", color="blue", fontsize=9)

# Ust sinir (mavi kare)
plt.plot(0, ust, "bs")
plt.text(0.15, ust + 2, f"Ust Sinir\n{ust:.2f}", color="blue", fontsize=9)

# Aykiri degerler (siyah daire)
for i, val in enumerate(veriSeti["horsepower"]):                     # enumerate(): Her degeri indeks numarasiyla birlikte gezer
    if val > ust or val < alt:                                       # Eger deger sinir disindaysa -> aykiri degerdir
        plt.plot(0, val, "ko")                                       # Aykiri degeri siyah daire olarak ciz
        x_offset = 0.2 if i % 2 == 0 else 0.25                      # Yazilar ust uste binmesin diye konum kaydirma
        plt.text(x_offset, val, f"{val:.1f}", fontsize=8, color="black")

plt.ylabel("Horsepower")                                            # Y ekseninin etiketini belirler
plt.xticks([])                                                       # X eksenindeki isaretleri gizler (tek degisken oldugu icin gerekmez)
plt.show()

In [ ]:
# 4) Aykiri degerleri kaldirmak icin asagidaki satirlari yorumdan cikarabilirsiniz
# veriSeti.drop(index=ust_aykiriDegerInd, inplace=True)             # drop(index=...): Belirtilen indekslerdeki satirlari siler
# veriSeti.drop(index=alt_aykiriDegerInd, inplace=True)             # inplace=True: Silme islemini dogrudan veriSeti uzerinde uygular

# **2.9 Ornekleme (Sampling ve Stratified Sampling)**

## 2.9.1. Egitim ve Test Veri Seti Olusturma
Egitim ve test veri seti olusturma, model egitimi ve dogrulama islemleri icin veriyi ayirma islemidir.
 - Genellikle verinin **%70'i egitim seti**, **%30'u ise test seti** olarak ayrilir.
 - Egitim seti, makine ogrenmesi modelinin egitilmesi icin kullanilirken, test seti modelin dogrulugunu ve basarisini degerlendirmek icin kullanilir.

In [ ]:
egitim = veriSeti.sample(frac=0.7, replace=False, random_state=1)   # sample(): Veri setinden rastgele satirlar secer
      # frac=0.7 -> veri setinin %70'ini sec (0.7 = %70 demektir)
      # replace=False -> ayni satir birden fazla kez secilmesin (tekrarsiz secim)
      # random_state=1 -> her calistirmada ayni satirlarin secilmesini saglar (tekrarlanabilirlik icin)

ind = veriSeti.index.isin(egitim.index)                             # isin(): Egitim setindeki satirlarin indekslerini ana veri setinde arar, True/False listesi dondurur

test = veriSeti[~ind]                                                # ~ind: True'lari False, False'lari True yapar -> egitim setinde OLMAYAN satirlari secer (test seti)

print("Egitim Seti Boyutu:", egitim.shape)                          # Egitim setinin satir ve sutun sayisini gosterir
print("Test Seti Boyutu:", test.shape)                              # Test setinin satir ve sutun sayisini gosterir

## 2.9.2. Tabakali Ornekleme (Stratified Sampling)
Tabakali ornekleme, verinin belirli gruplara (tabakalara) ayrilarak her gruptan orantili bir sekilde ornekleme yapilmasidir. Ozellikle dengesiz veri setlerinde, her sinifin dogru bir sekilde temsil edilmesini saglar.

- Tum veri setindeki sinif orani -> egitim ve test setine esit sekilde dagilir.
- Model egitiminde azinlik siniflarin ihmal edilmesi onlenir.
- Egitim ve test setlerinde "durum" degiskeninin dagilimi korunur.

In [ ]:
egitim1, test1 = train_test_split(                                  # train_test_split(): Veri setini otomatik olarak egitim ve test setine ayirir
    veriSeti,                                                       # Bolunecek veri seti
    train_size=0.7,                                                 # train_size=0.7: Verinin %70'i egitim, %30'u test olacak
    stratify=veriSeti["durum"],                                     # stratify: "durum" sutunundaki sinif oranlarini egitim/test setlerinde de korur
    random_state=1                                                  # random_state=1: Her calistirmada ayni bolunmeyi garantiler
)

print("\nDurum Dagilimi (Tum veri):\n", veriSeti.durum.value_counts())    # Tum veri setindeki sinif dagilimini gosterir
print("\nDurum Dagilimi (Egitim seti):\n", egitim1.durum.value_counts())  # Egitim setindeki sinif dagilimi (oranlar korunmus olmali)
print("\nDurum Dagilimi (Test seti):\n", test1.durum.value_counts())      # Test setindeki sinif dagilimi (oranlar korunmus olmali)

# **2.10 Yapay Kodlama (Label and Dummy Coding) ve Veri Normalizasyonu**

### 2.10.1. Kategorik Degiskeni Sayisal Forma Donusturme (cat.codes ve get_dummies)
#### Neden Onemli?
Makine ogrenmesi ve istatistiksel modeller **sayisal verilerle calisir**. Bu yuzden kategorik degiskenleri sayisal forma cevirmemiz gerekir. Bu islem genellikle iki yontemle yapilir:

---

#### 1. `cat.codes` -> Label Encoding (Etiket Kodlama)

- Her kategoriye bir **sayisal deger** atanir.
- Tek bir sutun olusturur.
Ornek:

| durum   | durum_s1 |
|---------|----------|
| Dusuk   | 0        |
| Orta    | 1        |
| Yuksek  | 2        |

---

#### 2. `get_dummies()` -> Dummy (One-Hot) Encoding

- Her kategori icin **ayri bir sutun** olusturur.
- 0 ve 1 degerleri icerir.
Ornek:

| Dusuk | Orta | Yuksek |
|-------|------|--------|
| 1     | 0    | 0      |
| 0     | 1    | 0      |
| 0     | 0    | 1      |

---

#### Projede Neden Her Ikisi de Kullanildi?

Kodda hem `cat.codes` hem de `get_dummies()` kullanildi. Bu sayede:

- `durum_s1` sutunu: **etiket kodlama** icin,
- `Dusuk`, `Orta`, `Yuksek` sutunlari: **dummy kodlama** icin olusturulmus oldu.

**Hangisi kullanilacak?**
-> Modeliniz sirali bilgiye ihtiyac duymuyorsa dummy encoding daha guvenlidir.

---

Sonuc:
Kategorik veriyi sayisala cevirmek zorunludur. Ancak **nasil cevirecginiz**, kullandiginiz algoritmaya baglidir.

In [ ]:
# Kategorik degiskeni sayisal forma donusturme

# 1. Yontem: cat.codes (Etiket Kodlama)
veriSeti["durum_s1"] = veriSeti["durum"].cat.codes                  # cat.codes: Kategorik degiskeni sayilara cevirir (Dusuk=0, Orta=1, Yuksek=2)
print("durum_s1 deger dagilimi:", veriSeti["durum_s1"].value_counts())  # Her sayisal koddan kac tane oldugunu gosterir

# 2. Yontem: get_dummies() (Yapay/Dummy Kodlama)
if not {'Dusuk', 'Orta', 'Yuksek'}.issubset(veriSeti.columns):     # issubset(): Bu sutunlar zaten varsa tekrar eklememek icin kontrol eder
    durum_s2 = pd.get_dummies(veriSeti["durum"], dtype=int)         # get_dummies(): Her kategori icin ayri bir sutun olusturur (0 veya 1 degerli)
    veriSeti = pd.concat([veriSeti, durum_s2], axis=1)              # pd.concat(axis=1): Yeni sutunlari mevcut tablonun sagina ekler
    print("Yapay Kodlama ile eklenen sutunlar:", list(durum_s2.columns))  # Eklenen yeni sutun isimlerini yazdirir

display(veriSeti.head(5))                                           # Tablonun ilk 5 satirini gosterir (yeni sutunlari gorebilmek icin)

### 2.10.2. Min-Max Normalizasyonu (0-1 Arasina Sikistirma)
#### Neden Onemli?
Veri setindeki sayisal sutunlar farkli araliklarda olabilir (ornegin: **city_mpg 13-49** arasinda, **horsepower 48-288** arasinda olabilir).
Bu fark, ozellikle uzaklik/matris temelli modellerde (KNN, K-Means, Lojistik Regresyon vb.) bazi degiskenlerin modele daha fazla etki etmesine yol acar.

Bu nedenle, tum degiskenlerin ayni olcege getirilmesi gerekir.
Min-Max Normalizasyonu, her bir sayisal degeri 0 ile 1 arasina sikistirarak bu islemi yapar.

**Avantaji:**
- Degiskenlerin esit etki gucune sahip olmasini saglar.

- Ozellikle mesafe tabanli algoritmalar icin onerilir.

In [ ]:
# Min-Max Normalizasyonu (verileri 0-1 arasina sikistirma)

# Sayisal sutunlari seciyoruz
sayisal_sutunlar = ["city_mpg", "engine_size", "horsepower", "curb_weight", "peak_rpm", "normalized_losses"]
veri = veriSeti[sayisal_sutunlar]                                   # Sadece normalizasyon uygulanacak sayisal sutunlari aliyoruz

scaler = MinMaxScaler()                                             # MinMaxScaler: Her degeri 0 ile 1 arasina sikistiracak nesneyi olusturur
scaler.fit(veri)                                                    # fit(): Verinin minimum ve maksimum degerlerini ogrenir (henuz donusum yapmaz)
n_veriSeti = scaler.transform(veri)                                 # transform(): Ogrenilen min-max degerlerine gore veriyi 0-1 arasina donusturur
n_veriSeti = pd.DataFrame(n_veriSeti, columns=veri.columns)        # Sonucu tekrar DataFrame (tablo) formatina cevirir, sutun isimlerini korur
print("\nMin-Max Normalizasyon sonrasi ilk 5 satir:\n", n_veriSeti.head())  # head(): Ilk 5 satiri gosterir, tum degerler artik 0-1 arasinda olmali